# GPU Sanity Check

Quick reference for the current GPU's capabilities — useful when switching machines or GPUs.

In [1]:
import pycuda.driver as cuda
cuda.init()

print(f"CUDA driver version : {cuda.get_driver_version()}")
print(f"Number of devices   : {cuda.Device.count()}")

CUDA driver version : 13030
Number of devices   : 1


## Device Properties

In [2]:
device = cuda.Device(0)
attrs  = device.get_attributes()
da     = cuda.device_attribute

print(f"Device name              : {device.name()}")
print(f"Compute capability       : {device.compute_capability()}")
print(f"Total memory             : {device.total_memory() / 1024**3:.2f} GB")
print()
print(f"Max threads per block    : {attrs[da.MAX_THREADS_PER_BLOCK]}")
print(f"Max block dims (x, y, z) : {attrs[da.MAX_BLOCK_DIM_X]}, {attrs[da.MAX_BLOCK_DIM_Y]}, {attrs[da.MAX_BLOCK_DIM_Z]}")
print(f"Max grid  dims (x, y, z) : {attrs[da.MAX_GRID_DIM_X]}, {attrs[da.MAX_GRID_DIM_Y]}, {attrs[da.MAX_GRID_DIM_Z]}")
print()
print(f"Warp size                : {attrs[da.WARP_SIZE]}")
print(f"Multiprocessors (SM)     : {attrs[da.MULTIPROCESSOR_COUNT]}")
print(f"Max threads per SM       : {attrs[da.MAX_THREADS_PER_MULTIPROCESSOR]}")
print(f"Shared mem per block     : {attrs[da.MAX_SHARED_MEMORY_PER_BLOCK] / 1024:.1f} KB")
print(f"Constant memory          : {attrs[da.TOTAL_CONSTANT_MEMORY] / 1024:.1f} KB")
print(f"L2 cache size            : {attrs[da.L2_CACHE_SIZE] / 1024:.1f} KB")
print(f"Memory clock rate        : {attrs[da.MEMORY_CLOCK_RATE] / 1e6:.1f} GHz")
print(f"Memory bus width         : {attrs[da.GLOBAL_MEMORY_BUS_WIDTH]} bits")

Device name              : NVIDIA RTX A3000 12GB Laptop GPU
Compute capability       : (8, 6)
Total memory             : 11.63 GB

Max threads per block    : 1024
Max block dims (x, y, z) : 1024, 1024, 64
Max grid  dims (x, y, z) : 2147483647, 65535, 65535

Warp size                : 32
Multiprocessors (SM)     : 32
Max threads per SM       : 1536
Shared mem per block     : 48.0 KB
Constant memory          : 64.0 KB
L2 cache size            : 3072.0 KB
Memory clock rate        : 7.0 GHz
Memory bus width         : 192 bits


## Safe Block Size Choices for 1D / 2D / 3D Kernels

Given `MAX_THREADS_PER_BLOCK` and `MAX_BLOCK_DIM_Z`, compute the largest safe `BLOCK_SIZE` per dimension.

In [3]:
import math

max_threads = attrs[da.MAX_THREADS_PER_BLOCK]
max_z       = attrs[da.MAX_BLOCK_DIM_Z]

bs_1d = max_threads
bs_2d = int(math.isqrt(max_threads))
bs_3d = min(int(max_threads ** (1/3)), max_z)

print(f"Recommended BLOCK_SIZE for 1D : {bs_1d}  ({bs_1d} threads/block)")
print(f"Recommended BLOCK_SIZE for 2D : {bs_2d}  ({bs_2d}x{bs_2d} = {bs_2d**2} threads/block)")
print(f"Recommended BLOCK_SIZE for 3D : {bs_3d}  ({bs_3d}x{bs_3d}x{bs_3d} = {bs_3d**3} threads/block)")

Recommended BLOCK_SIZE for 1D : 1024  (1024 threads/block)
Recommended BLOCK_SIZE for 2D : 32  (32x32 = 1024 threads/block)
Recommended BLOCK_SIZE for 3D : 10  (10x10x10 = 1000 threads/block)
